# H21a notebook counterpart: toy relative-performance crossover under prescribed imbalance

This notebook is the presentation and analysis companion to `run_experiment.py`. It intentionally **imports the script as the source of truth** instead of duplicating the training loop.

**Question.** As the prescribed first/last-layer imbalance factor $c$ grows, when does single-LR Muon stop achieving lower **held-out final training loss** than oracle per-layer SGD on this synthetic deep-linear toy problem?

**Scope.** This is a controlled optimizer-comparison toy study under reparameterization. It is useful as a **relative-performance crossover** test, but it is **not** a direct proof of the full direction-vs-scale mechanism.


## Study design and reporting choices

- **Model/task:** 4-layer deep linear regression on a synthetic Gaussian task family.
- **Methods compared:**
  - Muon with a single global LR and Newton-Schulz-normalized gradients.
  - Momentum SGD with an oracle-tuned LR for each layer.
- **Selection vs evaluation split:**
  - Hyperparameters are chosen on deterministic **selection seeds**.
  - Reported losses use fresh **held-out evaluation seeds**.
- **Primary metric:** held-out mean final training loss ratio `Muon / Oracle`.
- **Crossover language:** we report a **coarse bracket** (and optional local interpolation) when the held-out ratio crosses 1 on the tested grid.
- **Limits:** coarse grids, small seed counts, toy deep-linear setting, and final-loss comparison rather than direct mechanism decomposition.


In [ ]:
import os
from pathlib import Path
import importlib.util

import numpy as np
import matplotlib.pyplot as plt

try:
    from IPython.display import display, Markdown
except ImportError:
    def display(obj):
        print(obj)

    def Markdown(text):
        return text

try:
    import pandas as pd
except ImportError:
    pd = None

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
RELATIVE_SCRIPT = Path("experiments/Tier_2_Symmetry_Reparametrization_Tests/H21a_DIRECTION_SCALE_CROSSOVER/run_experiment.py")
REPO_ROOT_OVERRIDE = os.environ.get("MUON_RG_REPO_ROOT")


def find_repo_root():
    if REPO_ROOT_OVERRIDE:
        candidate = Path(REPO_ROOT_OVERRIDE).expanduser().resolve()
        if (candidate / RELATIVE_SCRIPT).exists():
            return candidate
        raise FileNotFoundError(
            f"MUON_RG_REPO_ROOT={REPO_ROOT_OVERRIDE!r} does not contain {RELATIVE_SCRIPT}"
        )

    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / RELATIVE_SCRIPT).exists():
            return base
    raise FileNotFoundError(
        f"Could not find repo root containing {RELATIVE_SCRIPT} starting from {Path.cwd()}; "
        "set MUON_RG_REPO_ROOT if running the notebook from outside the repo tree."
    )


REPO_ROOT = find_repo_root()
SCRIPT_PATH = REPO_ROOT / RELATIVE_SCRIPT
EXPERIMENT_DIR = SCRIPT_PATH.parent

spec = importlib.util.spec_from_file_location("h21a_direction_scale_crossover", SCRIPT_PATH)
h21a = importlib.util.module_from_spec(spec)
spec.loader.exec_module(h21a)

print(f"Repo root: {REPO_ROOT}")
print(f"Script path: {SCRIPT_PATH}")


## Reproducibility, execution mode, and expected workload

The notebook can be run in two modes:

- `full`: the scientific configuration matching the script defaults.
- `smoke`: a deliberately reduced configuration for quick path validation.

The environment variable `H21A_NOTEBOOK_MODE` controls the mode. This notebook defaults to `full`, but the smoke mode is what we use for automated basic checks because the exhaustive oracle grid is expensive.


In [ ]:
RUN_MODE = os.environ.get("H21A_NOTEBOOK_MODE", "full").strip().lower()
if RUN_MODE not in {"full", "smoke"}:
    raise ValueError(f"Unsupported H21A_NOTEBOOK_MODE={RUN_MODE!r}; use 'full' or 'smoke'.")

config = h21a.get_default_config() if RUN_MODE == "full" else h21a.make_smoke_config()
selection_seeds = h21a.get_selection_seeds(config)
evaluation_seeds = h21a.get_evaluation_seeds(config)
workload = h21a.estimate_workload(config)

config_lines = [
    f"**Run mode:** `{RUN_MODE}`",
    f"- network: {config['n_layers']}-layer deep linear {config['dim']}x{config['dim']}",
    f"- steps: {config['num_steps']} | batch size: {config['batch_size']}",
    f"- c values: {config['c_values']}",
    f"- Muon LR grid ({workload['muon_grid_size']} candidates): {config['muon_lr_grid']}",
    f"- Oracle per-layer grid ({len(config['per_layer_lr_grid'])}^{config['n_layers']} = {workload['oracle_grid_size']} candidates): {config['per_layer_lr_grid']}",
    f"- selection seeds: {selection_seeds}",
    f"- held-out evaluation seeds: {evaluation_seeds}",
    f"- train runs per c: selection {workload['selection_train_runs_per_c']} + held-out {workload['heldout_train_runs_per_c']} = {workload['total_train_runs_per_c']}",
    f"- total train runs: {workload['total_train_runs']}",
]

display(Markdown("\n".join(config_lines)))


## Execute the experiment through the script API

The cell below calls the script's `run_crossover_experiment(...)` function directly. In `full` mode this is intentionally expensive because the oracle baseline evaluates the entire per-layer LR grid.


In [ ]:
results = h21a.run_crossover_experiment(config=config, verbose=True)
summary_rows = h21a.build_summary_rows(results)
print(f"Completed {len(summary_rows)} c values in {RUN_MODE} mode.")


## Compact per-$c$ summary table

This table reports the held-out losses, the held-out ratio, the selected hyperparameters, and the divergence counts that now explicitly participate in hyperparameter ranking.


In [ ]:
table_rows = []
for row in summary_rows:
    table_rows.append({
        'c': row['c'],
        'muon_eval_mean_loss': row['muon_eval_mean_finite_loss'],
        'oracle_eval_mean_loss': row['oracle_eval_mean_finite_loss'],
        'muon_over_oracle': row['ratio'],
        'muon_best_lr': row['muon_best_lr'],
        'oracle_best_lrs': h21a.format_lr_list(row['oracle_best_lrs']),
        'muon_sel_diverged': row['muon_selection_diverged'],
        'oracle_sel_diverged': row['oracle_selection_diverged'],
        'muon_eval_diverged': row['muon_eval_diverged'],
        'oracle_eval_diverged': row['oracle_eval_diverged'],
    })

if pd is not None:
    summary_df = pd.DataFrame(table_rows)
    display(summary_df)
else:
    summary_df = table_rows
    for record in summary_df:
        print(record)


In [ ]:
crossover = results['crossover']
if crossover['status'] == 'exact_grid_hit':
    hit_text = ', '.join(f"c={item['c']:.0f}" for item in crossover['exact_grid_hits'])
    verdict = f"Observed an exact tested-grid hit of ratio ≈ 1 at {hit_text}."
elif crossover['status'] == 'bracketed':
    bracket = crossover['primary_bracket']
    interp = bracket['interpolated_c_star']
    interp_text = f" Local log-segment interpolation gives c ≈ {interp:.3g}." if interp is not None else ""
    verdict = (
        f"Observed a coarse in-range crossover bracket between c={bracket['left_c']:.0f} and "
        f"c={bracket['right_c']:.0f} because the held-out ratio moves from "
        f"{bracket['left_ratio']:.4g} to {bracket['right_ratio']:.4g}."
        f"{interp_text}"
    )
elif crossover['status'] == 'muon_better_all_tested':
    verdict = f"Muon had lower held-out loss for all tested c values, so any crossover would lie above c={config['c_values'][-1]:.0f}."
elif crossover['status'] == 'oracle_better_all_tested':
    verdict = f"Oracle had lower held-out loss for all tested c values, so any crossover would lie below c={config['c_values'][0]:.0f}."
else:
    verdict = "No clean adjacent crossover bracket could be stated from the tested held-out ratios."

display(Markdown(f"**Immediate reading.** {verdict}"))


## Figure 1: held-out ratio versus imbalance

The headline quantity is the held-out final-loss ratio `Muon / Oracle`. Values below 1 favor Muon; values above 1 favor the oracle per-layer SGD baseline.


In [ ]:
cs = np.array([row['c'] for row in summary_rows], dtype=float)
ratios = np.array([row['ratio'] for row in summary_rows], dtype=float)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(cs, ratios, marker='o', linewidth=2, label='Held-out Muon / Oracle loss')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1, label='Ratio = 1')

primary_bracket = results['crossover']['primary_bracket']
if primary_bracket is not None:
    ax.axvspan(primary_bracket['left_c'], primary_bracket['right_c'], alpha=0.15, color='tab:orange', label='Observed crossover bracket')

ax.set_xlabel('Imbalance factor c')
ax.set_ylabel('Held-out final loss ratio')
ax.set_title('Toy relative-performance crossover under prescribed imbalance')
ax.legend(loc='best')
plt.show()


In [ ]:
ratio_text = []
ratio_text.append("**Interpretation.** The ratio plot is the honest primary summary for this toy study: it tells us only which optimizer achieves lower held-out final loss after selection on separate seeds.")
ratio_text.append("A crossing of the horizontal line at 1 should be read as a coarse performance crossover on this task family, not as a direct decomposition of scale and direction effects.")
if results['crossover']['primary_bracket'] is not None:
    bracket = results['crossover']['primary_bracket']
    ratio_text.append(
        f"On the tested grid, the first adjacent sign change occurs between c={bracket['left_c']:.0f} and c={bracket['right_c']:.0f}."
    )
display(Markdown("\n\n".join(ratio_text)))


## Figure 2: absolute held-out final losses

Looking at the two held-out loss curves directly helps distinguish a genuine ratio crossover from cases where both methods are simultaneously very good or very poor.


In [ ]:
muon_losses = np.array([row['muon_eval_mean_finite_loss'] for row in summary_rows], dtype=float)
oracle_losses = np.array([row['oracle_eval_mean_finite_loss'] for row in summary_rows], dtype=float)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(cs, muon_losses, marker='o', linewidth=2, label='Muon held-out mean final loss')
ax.semilogx(cs, oracle_losses, marker='s', linewidth=2, label='Oracle held-out mean final loss')
ax.set_yscale('log')
ax.set_xlabel('Imbalance factor c')
ax.set_ylabel('Held-out mean final loss')
ax.set_title('Absolute held-out losses across prescribed imbalance')
ax.legend(loc='best')
plt.show()


In [ ]:
absolute_text = [
    "**Interpretation.** The absolute-loss view shows whether the ratio change is driven by Muon degrading, Oracle improving, or both.",
    "Because the metric is final training loss on a toy deep-linear task, these curves should be read as optimizer/task outcomes rather than as a universal performance claim.",
]
display(Markdown("\n\n".join(absolute_text)))


## Figure 3: selected learning rates

The left panel shows the best Muon scalar LR selected on the selection seeds. The right panel shows the oracle per-layer LR tuple selected at each $c$. These are diagnostics for how the prescribed imbalance changes the selected hyperparameters.


In [ ]:
muon_best_lrs = np.array([row['muon_best_lr'] for row in summary_rows], dtype=float)
oracle_best_lrs = np.array([row['oracle_best_lrs'] for row in summary_rows], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)

axes[0].semilogx(cs, muon_best_lrs, marker='o', linewidth=2)
axes[0].set_yscale('log')
axes[0].set_xlabel('Imbalance factor c')
axes[0].set_ylabel('Selected Muon LR')
axes[0].set_title('Best Muon scalar LR by c')

im = axes[1].imshow(np.log10(oracle_best_lrs), aspect='auto', cmap='viridis')
axes[1].set_xticks(range(config['n_layers']))
axes[1].set_xticklabels([f'Layer {i}' for i in range(config['n_layers'])])
axes[1].set_yticks(range(len(cs)))
axes[1].set_yticklabels([f'{c:g}' for c in cs])
axes[1].set_xlabel('Layer index')
axes[1].set_ylabel('Imbalance factor c')
axes[1].set_title('log10(selected oracle per-layer LR)')
fig.colorbar(im, ax=axes[1], label='log10(LR)')

plt.show()


In [ ]:
lr_text = [
    "**Interpretation.** These selected-LR diagnostics help separate outcome changes from hyperparameter-search behavior.",
    "The oracle panel is especially useful because it reveals whether the exhaustive per-layer search responds to larger prescribed imbalance by shifting LR mass across layers rather than by simply improving uniformly everywhere.",
]
display(Markdown("\n\n".join(lr_text)))


## Divergence counts and stability reporting

A key rigor fix in this pass is that diverged seeds are no longer silently dropped during LR selection. Candidates are ranked by **divergence count first** and **mean finite loss second**.


In [ ]:
divergence_rows = []
for row in summary_rows:
    divergence_rows.append({
        'c': row['c'],
        'muon_selection_diverged': row['muon_selection_diverged'],
        'oracle_selection_diverged': row['oracle_selection_diverged'],
        'muon_eval_diverged': row['muon_eval_diverged'],
        'oracle_eval_diverged': row['oracle_eval_diverged'],
        'muon_eval_losses': row['muon_eval_losses'],
        'oracle_eval_losses': row['oracle_eval_losses'],
    })

if pd is not None:
    display(pd.DataFrame(divergence_rows))
else:
    for record in divergence_rows:
        print(record)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.semilogx(cs, [row['muon_eval_diverged'] for row in summary_rows], marker='o', linewidth=2, label='Muon held-out diverged seeds')
ax.semilogx(cs, [row['oracle_eval_diverged'] for row in summary_rows], marker='s', linewidth=2, label='Oracle held-out diverged seeds')
ax.set_xlabel('Imbalance factor c')
ax.set_ylabel('Held-out divergence count')
ax.set_title('Held-out divergence counts by c')
ax.legend(loc='best')
plt.show()


In [ ]:
div_text = [
    "**Interpretation.** Divergence counts now participate directly in model selection, so a candidate that diverges on more seeds cannot win merely by having a low mean over the surviving runs.",
    "This does not remove all small-sample concerns, but it does prevent the most obvious instability masking present in the earlier version of the pair.",
]
display(Markdown("\n\n".join(div_text)))


## Final calibrated verdict

The conclusion below is intentionally narrow. It reports what this pair actually measures after the first rigor pass, and it states the main limitations explicitly.


In [ ]:
trend_fit = results['trend_fit']
final_lines = [
    "### What this notebook/script pair now supports",
    f"- Primary result: a held-out final-loss comparison between Muon and oracle per-layer SGD across prescribed imbalance values {config['c_values']}.",
    f"- Selection seeds: {results['selection_seeds']}",
    f"- Held-out evaluation seeds: {results['evaluation_seeds']}",
    f"- Workload: {results['workload']['total_train_runs']} training runs in `{RUN_MODE}` mode.",
]

if crossover['status'] == 'exact_grid_hit':
    final_lines.append(
        '- Crossover verdict: at least one tested grid point achieved a held-out ratio approximately equal to 1.'
    )
elif crossover['status'] == 'bracketed':
    bracket = crossover['primary_bracket']
    final_lines.append(
        f"- Crossover verdict: the tested grid brackets a toy performance crossover between c={bracket['left_c']:.0f} and c={bracket['right_c']:.0f}."
    )
    if bracket['interpolated_c_star'] is not None:
        final_lines.append(
            f"- Optional local interpolation within that bracket gives c ≈ {bracket['interpolated_c_star']:.3g}."
        )
elif crossover['status'] == 'muon_better_all_tested':
    final_lines.append(
        f"- Crossover verdict: no in-range crossing was observed; Muon had lower held-out loss for all tested c values up to {config['c_values'][-1]:.0f}."
    )
elif crossover['status'] == 'oracle_better_all_tested':
    final_lines.append(
        f"- Crossover verdict: no in-range crossing was observed; the oracle baseline had lower held-out loss for all tested c values starting at {config['c_values'][0]:.0f}."
    )
else:
    final_lines.append('- Crossover verdict: no clean adjacent bracket could be stated from the finite held-out ratios.')

if trend_fit is not None:
    final_lines.append(
        f"- Exploratory global trend fit: log10(ratio) = {trend_fit['slope']:.3f} log10(c) + {trend_fit['intercept']:.3f} (R^2={trend_fit['r_squared']:.3f})."
    )

final_lines.extend([
    '',
    '### Limitations that still matter',
    '- synthetic deep-linear toy problem rather than a realistic network/task pair',
    '- coarse c grid and coarse LR grids',
    '- small seed counts forced by the exhaustive oracle search cost',
    '- outcome is relative final training loss, not a direct proof of a full gauge-fixing or direction-vs-scale mechanism',
])

display(Markdown("\n".join(final_lines)))
